In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score

# Load data
train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

# Advanced Feature Engineering
for df in [train, test]:
    # Fix missing values
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
    
    # Encode
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
    
    # New features
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    
    # Title
    df["Title"] = df["Name"].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df["Title"] = df["Title"].map({
        "Mr": 0, "Miss": 1, "Mrs": 2,
        "Master": 3, "Dr": 4
    }).fillna(5)
    
    # Age groups
    df["AgeGroup"] = pd.cut(df["Age"],
                     bins=[0, 12, 18, 35, 60, 100],
                     labels=[0, 1, 2, 3, 4]).astype(int)
    
    # Fare per person
    df["FarePerPerson"] = df["Fare"] / df["FamilySize"]

# Features
features = ["Pclass", "Sex", "Age", "Fare",
            "FamilySize", "IsAlone", "Title",
            "AgeGroup", "FarePerPerson"]

X = train[features]
y = train["Survived"]
X_test = test[features]

# GridSearch for best parameters
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 5, 6],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy')
grid_search.fit(X, y)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

# Submit
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)
output = pd.DataFrame({
    'PassengerId': test.PassengerId,
    'Survived': predictions
})
output.to_csv('submission.csv', index=False)
print("Done! Submit now!")

Best Parameters: {'max_depth': 6, 'min_samples_split': 2, 'n_estimators': 100}
Best CV Score: 0.8350134957002071
Done! Submit now!


In [4]:
# Simple but powerful!
features = ["Pclass", "Sex", "SibSp", "Parch", "Fare"]

X = pd.get_dummies(train[features])
X_test = pd.get_dummies(test[features])

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=1)
model.fit(X, y)

predictions = model.predict(X_test)
output = pd.DataFrame({
    'PassengerId': test.PassengerId,
    'Survived': predictions
})
output.to_csv('submission.csv', index=False)
print("Done!")

Done!
